## 🚕 NYC YELLOW TAXI ENRICHMENT PIPELINE

---

### 📥 **Step 1: Read Data from Source Tables**

> In this stage, we read cleansed yellow taxi trip data and taxi zone lookup information from the Databricks tables.

---

In [0]:
df_trip_cleansed = spark.read.table("NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED")

In [0]:
df_zones = spark.read.table("NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP")

#### JOIN TRIPS WITH PICKUP ZONE DETAILS TO OBTAIN BOROUGH AND ZONE NAME
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED
- NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP

In [0]:
df_taxi_cleasned_df_lookup = df_trip_cleansed.join(
                df_zones, 
                df_trip_cleansed.pu_location_id == df_zones.location_id,
                "left"
                ).select(
                    df_trip_cleansed.vendor,
                    df_trip_cleansed.tpep_pickup_datetime,
                    df_trip_cleansed.tpep_dropoff_datetime,
                    df_trip_cleansed.trip_duration,
                    df_trip_cleansed.passenger_count,
                    df_trip_cleansed.trip_distance,
                    df_trip_cleansed.rate_type,
                    df_zones.borough.alias("pu_borough"),   # pickup borough
                    df_zones.zone.alias("pu_zone"),         # pickup zone
                    df_trip_cleansed.do_location_id,                # dropoff location ID for next join
                    df_trip_cleansed.payment_type,
                    df_trip_cleansed.fare_amount,
                    df_trip_cleansed.extra,
                    df_trip_cleansed.mta_tax,
                    df_trip_cleansed.tolls_amount,
                    df_trip_cleansed.improvement_surcharge,
                    df_trip_cleansed.total_amount,
                    df_trip_cleansed.congestion_surcharge,
                    df_trip_cleansed.airport_fee,  
                    df_trip_cleansed.cbd_congestion_fee,
                    df_trip_cleansed.load_timestamp
                )

#### JOIN TRIPS WITH PICKUP ZONE DETAILS TO OBTAIN ENRICHMENT
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED
- NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP

In [0]:
df_cleansed_lookup_trans = df_taxi_cleasned_df_lookup.join(
                                df_zones, 
                                df_taxi_cleasned_df_lookup.do_location_id == df_zones.location_id,
                                "left"
                                ).select(
                                            df_taxi_cleasned_df_lookup.vendor,
                                            df_taxi_cleasned_df_lookup.tpep_pickup_datetime,
                                            df_taxi_cleasned_df_lookup.tpep_dropoff_datetime,
                                            df_trip_cleansed.trip_duration,
                                            df_taxi_cleasned_df_lookup.passenger_count,
                                            df_taxi_cleasned_df_lookup.trip_distance,
                                            df_taxi_cleasned_df_lookup.rate_type,
                                            df_taxi_cleasned_df_lookup.pu_borough,
                                            df_zones.borough.alias("do_borough"), # dropoff borough
                                            df_taxi_cleasned_df_lookup.pu_zone,
                                            df_zones.zone.alias("do_zone"),       # dropoff zone
                                            df_taxi_cleasned_df_lookup.payment_type,
                                            df_taxi_cleasned_df_lookup.fare_amount,
                                            df_taxi_cleasned_df_lookup.extra,
                                            df_taxi_cleasned_df_lookup.mta_tax,
                                            df_taxi_cleasned_df_lookup.tolls_amount,
                                            df_taxi_cleasned_df_lookup.improvement_surcharge,
                                            df_taxi_cleasned_df_lookup.total_amount,
                                            df_taxi_cleasned_df_lookup.congestion_surcharge,
                                            df_taxi_cleasned_df_lookup.airport_fee,  
                                            df_taxi_cleasned_df_lookup.cbd_congestion_fee,
                                            df_taxi_cleasned_df_lookup.load_timestamp
                                )

#### LOAD THE DATA INTO THE SILVER TABLE
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED

In [0]:
df_cleansed_lookup_trans.write.mode("overwrite").saveAsTable("NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED")

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED')